#Initialisation

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType,DateType
from pyspark.sql.functions import col,trim,length

##Read Bronze table

In [0]:
df=spark.table("workspace.bronze.crm_sales_details")

#silver transformation


#Trimming

In [0]:
for field in df.schema.fields:
    if field.dataType==StringType():
        df=df.withColumn(field.name,trim((col(field.name))))

#Date formating

In [0]:

df = (
    df
    .withColumn(
        "sls_order_dt",
        F.when(
            (col("sls_order_dt") == 0) | (length(col("sls_order_dt")) != 8),
            None
        ).otherwise(F.to_date(col("sls_order_dt").cast("string"), "yyyyMMdd"))
    )
    .withColumn(
        "sls_ship_dt",
        F.when(
            (col("sls_ship_dt") == 0) | (length(col("sls_ship_dt")) != 8),
            None
        ).otherwise(F.to_date(col("sls_ship_dt").cast("string"), "yyyyMMdd"))
    )
    .withColumn(
        "sls_due_dt",
        F.when(
            (col("sls_due_dt") == 0) | (length(col("sls_due_dt")) != 8),
            None
        ).otherwise(F.to_date(col("sls_due_dt").cast("string"), "yyyyMMdd"))
    )
)

In [0]:
df.select("sls_ship_dt").show()

##sales and price correction

In [0]:


df = (
    df
    .withColumn(
        "sls_price",
        F.when(
            (col("sls_price").isNull()) | (col("sls_price") <= 0),
            F.when(
                col("sls_quantity") != 0,
                col("sls_sales") / col("sls_quantity")
            ).otherwise(None)
        ).otherwise(col("sls_price"))
    )
)

##Renaming the column

In [0]:
RenameMap={
    "sls_ord_num":"Order_number",
    "sls_prd_key":"Product_key",
    "sls_cust_id":"Customer_id",
    "sls_order_dt":"Order_date",
    "sls_ship_dt":"Ship_date",
    "sls_due_dt":"Due_date",
    "sls_sales":"Sales",
    "sls_quantity":"Quantity",
    "sls_price":"Price"
}
for oldname,newname in RenameMap.items():
    df=df.withColumnRenamed(oldname,newname)

In [0]:
df.limit(10).display()

##Writing to solver table

In [0]:
df.write.mode("overwrite").format("delta").saveAsTable("silver.crm_sales")

In [0]:

%sql
SELECT * FROM workspace.silver.crm_sales LIMIT 10